In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Ανάλυση Δεδομένων Τρεξίματος από Garmin Forerunner 245\n",
    "\n",
    "Σκοπός αυτού του notebook είναι να αναλύσει δεδομένα τρεξίματος που έχουν εξαχθεί από ένα ρολόι\n",
    "Garmin Forerunner 245 και αποθηκευτεί σε μια βάση δεδομένων SQLite χρησιμοποιώντας το project garmindb.\n",
    "\n",
    "Θα πραγματοποιήσουμε τις ακόλουθες αναλύσεις:\n",
    "1. Βασικές Μετρήσεις και Τάσεις (Ρυθμός, Απόσταση, Διάρκεια)\n",
    "2. Ανάλυση Καρδιακών Παλμών (Μέσος Παλμός, Ζώνες Παλμών)\n",
    "3. Ανάλυση Ρυθμού και Υψομέτρου\n",
    "4. Στατιστικά Laps (αν υπάρχουν)\n",
    "5. Μακροπρόθεσμες Τάσεις και Συγκρίσεις"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Εισαγωγή Βιβλιοθηκών"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sqlite3\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "# Ρυθμίσεις για καλύτερες οπτικοποιήσεις\n",
    "plt.style.use('seaborn-v0_8-whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (12, 6)\n",
    "plt.rcParams['font.size'] = 12"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Σύνδεση στη Βάση Δεδομένων SQLite"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# **ΠΡΟΣΟΧΗ:** Αντικαταστήστε την παρακάτω διαδρομή με την πραγματική διαδρομή προς το αρχείο της βάσης δεδομένων σας.\n",
    "DATABASE_PATH = 'path/to/your/garmin_data.db'\n",
    "\n",
    "def connect_to_db(db_path):\n",
    "    \"\"\"Συνδέεται με τη βάση δεδομένων SQLite.\"\"\"\n",
    "    try:\n",
    "        conn = sqlite3.connect(db_path)\n",
    "        return conn\n",
    "    except sqlite3.Error as e:\n",
    "        print(f\"Σφάλμα σύνδεσης με τη βάση δεδομένων: {e}\")\n",
    "        return None\n",
    "\n",
    "conn = connect_to_db(DATABASE_PATH)\n",
    "\n",
    "if conn:\n",
    "    cursor = conn.cursor()\n",
    "    print(f\"Επιτυχής σύνδεση με τη βάση δεδομένων: {DATABASE_PATH}\")\n",
    "else:\n",
    "    exit()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Βασικές Μετρήσεις και Τάσεις"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Μέσος Ρυθμός ανά Τρέξιμο"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "query_avg_pace = \"\"\"\n",
    "SELECT\n",
    "    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,\n",
    "    AVG(1000.0 / r.speed) / 60.0 AS avg_pace_min_km\n",
    "FROM activity a\n",
    "JOIN record r ON a.activity_id = r.activity_id\n",
    "WHERE a.sport = 'running' AND r.speed > 0\n",
    "GROUP BY run_date\n",
    "ORDER BY run_date;\n",
    "\"\"\"\n",
    "df_pace = pd.read_sql_query(query_avg_pace, conn, parse_dates=['run_date'])\n",
    "\n",
    "if not df_pace.empty:\n",
    "    plt.figure(figsize=(12, 6))\n",
    "    plt.plot(df_pace['run_date'], df_pace['avg_pace_min_km'], marker='o', linestyle='-')\n",
    "    plt.title('Μέσος Ρυθμός Τρεξίματος με την Πάροδο του Χρόνου')\n",
    "    plt.xlabel('Ημερομηνία')\n",
    "    plt.ylabel('Μέσος Ρυθμός (λεπτά/km)')\n",
    "    plt.grid(True)\n",
    "    plt.xticks(rotation=45, ha='right')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"Δεν βρέθηκαν δεδομένα τρεξίματος για ανάλυση ρυθμού.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Μέση Απόσταση ανά Τρέξιμο"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "query_distance = \"\"\"\n",
    "SELECT\n",
    "    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,\n",
    "    a.total_distance / 1000.0 AS distance_km\n",
    "FROM activity a\n",
    "WHERE a.sport = 'running' AND a.total_distance > 0\n",
    "ORDER BY run_date;\n",
    "\"\"\"\n",
    "df_distance = pd.read_sql_query(query_distance, conn, parse_dates=['run_date'])\n",
    "\n",
    "if not df_distance.empty:\n",
    "    plt.figure(figsize=(12, 6))\n",
    "    plt.bar(df_distance['run_date'], df_distance['distance_km'])\n",
    "    plt.title('Απόσταση Τρεξίματος ανά Ημερομηνία')\n",
    "    plt.xlabel('Ημερομηνία')\n",
    "    plt.ylabel('Απόσταση (km)')\n",
    "    plt.xticks(rotation=45, ha='right')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"Δεν βρέθηκαν δεδομένα τρεξίματος για ανάλυση απόστασης.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Μέση Διάρκεια ανά Τρέξιμο"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "query_duration = \"\"\"\n",
    "SELECT\n",
    "    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,\n",
    "    a.total_duration / 60.0 AS duration_minutes\n",
    "FROM activity a\n",
    "WHERE a.sport = 'running' AND a.total_duration > 0\n",
    "ORDER BY run_date;\n",
    "\"\"\"\n",
    "df_duration = pd.read_sql_query(query_duration, conn, parse_dates=['run_date'])\n",
    "\n",
    "if not df_duration.empty:\n",
    "    plt.figure(figsize=(12, 6))\n",
    "    plt.plot(df_duration['run_date'], df_duration['duration_minutes'], marker='o', linestyle='-')\n",
    "    plt.title('Διάρκεια Τρεξίματος με την Πάροδο του Χρόνου')\n",
    "    plt.xlabel('Ημερομηνία')\n",
    "    plt.ylabel('Διάρκεια (λεπτά)')\n",
    "    plt.grid(True)\n",
    "    plt.xticks(rotation=45, ha='right')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"Δεν βρέθηκαν δεδομένα τρεξίματος για ανάλυση διάρκειας.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Ανάλυση Καρδιακών Παλμών"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Μέσος Καρδιακός Παλμός ανά Τρέξιμο"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "query_avg_hr = \"\"\"\n",
    "SELECT\n",
    "    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,\n",
    "    AVG(r.heart_rate) AS avg_heart_rate\n",
    "FROM activity a\n",
    "JOIN record r ON a.activity_id = r.activity_id\n",
    "WHERE a.sport = 'running' AND r.heart_rate > 0\n",
    "GROUP BY run_date\n",
    "ORDER BY run_date;\n",
    "\"\"\"\n",
    "df_hr = pd.read_sql_query(query_avg_hr, conn, parse_dates=['run_date'])\n",
    "\n",
    "if not df_hr.empty:\n",
    "    plt.figure(figsize=(12, 6))\n",
    "    plt.plot(df_hr['run_date'], df_hr['avg_heart_rate'], marker='o', linestyle='-')\n",
    "    plt.title('Μέσος Καρδιακός Παλμός ανά Τρέξιμο')\n",
    "    plt.xlabel('Ημερομηνία')\n",
    "    plt.ylabel('Μέσος Καρδιακός Παλμός (bpm)')\n",
    "    plt.grid(True)\n",
    "    plt.xticks(rotation=45, ha='right')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"Δεν βρέθηκαν δεδομένα καρδιακών παλμών για τρέξιμο.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Κατανομή Χρόνου σε Ζώνες Καρδιακών Παλμών"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def calculate_hr_zones(heart_rates, max_hr):\n",
    "    \"\"\"Υπολογίζει το ποσοστό του χρόνου που πέρασε σε κάθε ζώνη καρδιακών παλμών.\"\"\"\n",
    "    zones = {'Zone 1': 0, 'Zone 2': 0, 'Zone 3': 0, 'Zone 4': 0, 'Zone 5': 0}\n",
    "    total_time = len(heart_rates)\n",
    "    if total_time > 0:\n",
    "        for hr in heart_rates:\n",
    "            if hr < 0.6 * max_hr:\n",
    "                zones['Zone 1'] += 1\n",
    "            elif hr < 0.7 * max_hr:\n",
    "                zones['Zone 2'] += 1\n",
    "            elif hr < 0.8 * max_hr:\n",
    "                zones['Zone 3'] += 1\n",
    "            elif hr < 0.9 * max_hr:\n",
    "                zones['Zone 4'] += 1\n",
    "            else:\n",
    "                zones['Zone 5'] += 1\n",
    "        return {zone: count / total_time for zone, count in zones.items()}\n",
    "    else:\n",
    "        return zones\n",
    "\n",
    "# **ΠΡΟΣΟΧΗ:** Αντικαταστήστε το παρακάτω με τον εκτιμώμενο μέγιστο καρδιακό παλμό σας.\n",
    "MAX_HEART_RATE = 190\n",
    "\n",
    "query_hr_records = \"\"\"\n",
    "SELECT\n",
    "    CAST(strftime('%Y-%m-%d', a.start_time) AS TEXT) AS run_date,\n",
    "    r.heart_rate\n",
    "FROM activity a\n",
    "JOIN record r ON a.activity_id = r.activity_id\n",
    "WHERE a.sport = 'running' AND r.heart_rate > 0\n",
    "ORDER BY a.start_time;\n",
    "\"\"\"\n",
    "df_hr_records = pd.read_sql_query(query_hr_records, conn)\n",
    "\n",
    "if not df_hr_records.empty:\n",
    "    hr_zone_analysis = df_hr_records.groupby('run_date')['heart_rate'].apply(list).apply(lambda x: calculate_hr_zones(x, MAX_HEART_RATE)).apply(pd.Series)\n",
    "    hr_zone_analysis.index = pd.to_datetime(hr_zone_analysis.index)\n",
    "    hr_zone_analysis = hr_zone_analysis.sort_index()\n",
    "\n",
    "    hr_zone_analysis.plot(kind='bar', stacked=True, figsize=(14, 7))\n",
    "    plt.title('Κατανομή Χρόνου σε Ζώνες Καρδιακών Παλμών ανά Τρέξιμο')\n",
    "    plt.xlabel('Ημερομηνία')\n",
    "    plt.ylabel('Ποσοστό Χρόνου')\n",
    "    plt.xticks(rotation=45, ha='right')\n",
    "    plt.legend(title='Ζώνη Καρδιακών Παλμών')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "else:\n",
    "    print(\"Δεν βρέθηκαν λεπτομερή δεδομένα καρδιακών παλμών για ανάλυση ζωνών.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Ανάλυση Ρυθμού και Υψομέτρου"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Μεταβολή Ρυθ